# Contract Agent Demo - ESA/Contract Ingestion

This notebook demonstrates the Contract Agent for processing and analyzing contracts using LangGraph and IBM Watsonx.

## Features
- Document reading (read-only access)
- OCR and text extraction
- Metadata extraction (parties, dates, terms, obligations)
- Vector database ingestion
- Semantic search capabilities

## 1. Setup and Imports

In [1]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

# Add agents directory to path
sys.path.insert(0, os.path.join(os.getcwd(), 'agents'))

from contract_agent import ContractAgent

# Load environment variables
load_dotenv()

print("Imports successful")

Imports successful


## 2. Initialize Contract Agent

In [2]:
# Check for credentials
apikey = os.getenv("WATSONX_APIKEY")
project_id = os.getenv("WATSONX_PROJECT_ID")

if not apikey or not project_id:
    print("Error: Missing Watsonx credentials in .env file")
    print("\nPlease create a .env file based on .env.example with your credentials:")
    print("  - WATSONX_APIKEY")
    print("  - WATSONX_PROJECT_ID")
else:
    print("Credentials loaded successfully")

# Initialize agent
agent = ContractAgent(
    apikey=apikey,
    project_id=project_id,
    model_id=os.getenv("MODEL_ID", "ibm/granite-13b-chat-v2"),
    embedding_model_id=os.getenv("EMBEDDING_MODEL_ID", "ibm/granite-embedding-278m-multilingual"),
    vector_store_path=os.getenv("VECTOR_STORE_PATH", "./contract_vector_store")
)

print("   Contract Agent initialized")
print(f"   Model: {agent.model_id}")
print(f"   Embedding Model: {agent.embedding_model_id}")
print(f"   Vector Store: {agent.vector_store_path}")

Credentials loaded successfully
   Contract Agent initialized
   Model: mistralai/mistral-medium-2505
   Embedding Model: ibm/granite-embedding-278m-multilingual
   Vector Store: ./contract_vector_store


## 3. Find Available Contract Documents

In [3]:
from pathlib import Path

docs_dir = Path("docs")

if not docs_dir.exists():
    print(f"Error: docs directory not found at {docs_dir.resolve()}")
else:
    # Get list of contract files
    contract_files = (
        list(docs_dir.glob("*.docx")) +
        list(docs_dir.glob("*.xlsx")) +
        list(docs_dir.glob("*.txt"))
    )
    
    if not contract_files:
        print(f"Error: No contract files found in {docs_dir.resolve()}")
        print("   Supported formats: .docx, .xlsx, .txt")
    else:
        print(f"Found {len(contract_files)} contract file(s):\n")
        for i, file in enumerate(contract_files, 1):
            print(f"   {i}. {file.resolve()}")

Found 5 contract file(s):

   1. /Users/arpit/Desktop/build engineering/sales-demo/agents/docs/Confluent_IBM-1.30.2025.docx
   2. /Users/arpit/Desktop/build engineering/sales-demo/agents/docs/Confluent_IBM-1.30.2024.docx
   3. /Users/arpit/Desktop/build engineering/sales-demo/agents/docs/Confluent_IBM-3.29.2024.docx
   4. /Users/arpit/Desktop/build engineering/sales-demo/agents/docs/Confluent_IBM-5.30.2023.docx
   5. /Users/arpit/Desktop/build engineering/sales-demo/agents/docs/Confluent Sales Cloud Infor.xlsx


## 4. Process First Contract

This demonstrates the full LangGraph workflow:
1. Read Document (read-only)
2. Extract Metadata (OCR + LLM)
3. Normalize Structure
4. Ingest to Vector DB
5. Generate Summary

In [4]:
# Process first contract
if contract_files:
    contract_file = contract_files[0]
    full_path = contract_file.resolve()
    
    print(f"Processing contract: {contract_file.name}")
    print(f"Full path: {full_path}")
    print("-" * 70)
    print("\nRunning Contract Agent workflow...")
    print("   Step 1: Reading document (read-only access)...")
    print("   Step 2: Extracting metadata (OCR/text extraction)...")
    print("   Step 3: Normalizing contract data...")
    print("   Step 4: Ingesting into vector database...")
    print("   Step 5: Generating structured summary...\n")
    
    try:
        # Run the agent with absolute path
        result = agent.run(str(full_path))
        
        print("Contract processing complete!\n")
        print(result["generated_text"])
        
    except Exception as e:
        print(f"Error during processing: {str(e)}")
        import traceback
        traceback.print_exc()
else:
    print("No contract files available to process")

Processing contract: Confluent_IBM-1.30.2025.docx
Full path: /Users/arpit/Desktop/build engineering/sales-demo/agents/docs/Confluent_IBM-1.30.2025.docx
----------------------------------------------------------------------

Running Contract Agent workflow...
   Step 1: Reading document (read-only access)...
   Step 2: Extracting metadata (OCR/text extraction)...
   Step 3: Normalizing contract data...
   Step 4: Ingesting into vector database...
   Step 5: Generating structured summary...

DEBUG: Reading document from: /Users/arpit/Desktop/build engineering/sales-demo/agents/docs/Confluent_IBM-1.30.2025.docx
DEBUG: Extracted 11820 characters
DEBUG: Returning raw_text with 11820 characters


/Users/arpit/Desktop/build engineering/sales-demo/agents/contract_agent.py:104: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  self.vector_store = Chroma(


Contract processing complete!

CONTRACT ANALYSIS SUMMARY

File: /Users/arpit/Desktop/build engineering/sales-demo/agents/docs/Confluent_IBM-1.30.2025.docx
Document Length: 11820 characters

EXTRACTED METADATA:
----------------------------------------------------------------------
 Here's the extracted metadata:

PARTIES: IBM entity and BP (Business Partner)
EFFECTIVE_DATE: Jan 31, 2025
TERM_LENGTH: Not specified
KEY_OBLIGATIONS:
- BP will receive the right to use specified Cloud Services in accordance with the Agreement, relevant Service Description(s), and this TD.
- BP will pay the applicable payment as described for the Cloud Services acquired.
- BP must either renew its Subscription or cancel it upon expiration.
- Subscriptions require a committed usage level over a selected subscription period for eligible Cloud Services.
- Charges for the usage of selected eligible Cloud Services are deducted from the committed subscription usage level in a current cycle.
- Additional usage in a 

## 5. Semantic Search - Query 1: Parties Involved

In [5]:
query = "What are the parties involved in this contract?"
print(f"Query: {query}")
print("-" * 70)

try:
    results = agent.query_contracts(query, k=2)
    
    if results:
        for i, doc in enumerate(results, 1):
            print(f"\nResult {i}:")
            content = doc.page_content.strip()
            preview = content[:300] + "..." if len(content) > 300 else content
            print(preview)
            
            if doc.metadata:
                print(f"\nMetadata: {doc.metadata.get('file_path', 'N/A')}")
    else:
        print("No results found")
        
except Exception as e:
    print(f"Query error: {str(e)}")

Query: What are the parties involved in this contract?
----------------------------------------------------------------------

Result 1:
) between the parties and is the parties' complete agreement and replaces all prior oral or written communications between the parties regarding the transactions described in this ESA TD. All terms used in this ESA TD and not otherwise defined herein shall have the meanings ascribed to such terms in...

Metadata: /Users/arpit/Desktop/build engineering/sales-demo/agents/docs/Confluent_IBM-1.30.2025.docx

Result 2:
) between the parties and is the parties' complete agreement and replaces all prior oral or written communications between the parties regarding the transactions described in this ESA TD. All terms used in this ESA TD and not otherwise defined herein shall have the meanings ascribed to such terms in...

Metadata: /Users/arpit/Desktop/build engineering/sales-demo/agents/docs/Confluent_IBM-1.30.2024.docx


## 6. Semantic Search - Query 2: Effective Date and Term

In [6]:
query = "What is the effective date and term length?"
print(f"Query: {query}")
print("-" * 70)

try:
    results = agent.query_contracts(query, k=2)
    
    if results:
        for i, doc in enumerate(results, 1):
            print(f"\nResult {i}:")
            content = doc.page_content.strip()
            preview = content[:300] + "..." if len(content) > 300 else content
            print(preview)
    else:
        print("No results found")
        
except Exception as e:
    print(f"Query error: {str(e)}")

Query: What is the effective date and term length?
----------------------------------------------------------------------

Result 1:
nt new functionality or capability and does not constitute Value Add.
Term
The term of this TD will be Three (3) years from the Effective Date (“Initial Term”).
Invoicing and Ordering
Initial Invoice – only applies to Subscriptions
BP is placing the following order for the Cloud Services subscriptio...

Result 2:
nt new functionality or capability and does not constitute Value Add.
Term
The term of this TD will be Two (2) years from the Effective Date (“Initial Term”).
Invoicing and Ordering
Initial Invoice – only applies to Subscriptions
BP is placing the following order for the Cloud Services subscription(...


## 7. Semantic Search - Query 3: Key Obligations

In [7]:
query = "What are the key obligations and milestones?"
print(f"Query: {query}")
print("-" * 70)

try:
    results = agent.query_contracts(query, k=2)
    
    if results:
        for i, doc in enumerate(results, 1):
            print(f"\nResult {i}:")
            content = doc.page_content.strip()
            preview = content[:300] + "..." if len(content) > 300 else content
            print(preview)
    else:
        print("No results found")
        
except Exception as e:
    print(f"Query error: {str(e)}")

Query: What are the key obligations and milestones?
----------------------------------------------------------------------

Result 1:
nt new functionality or capability and does not constitute Value Add.
Term
The term of this TD will be Three (3) years from the Effective Date (“Initial Term”).
Invoicing and Ordering
Initial Invoice – only applies to Subscriptions
BP is placing the following order for the Cloud Services subscriptio...

Result 2:
ervices for which BP provides services, (ii) an escalation process, (iii) identification and maintenance of qualified technical support personnel and mutually-agreed resource commitments, management contacts, and support location(s), and (iv) logging and reporting procedures for BP's service activit...


## 8. Semantic Search - Query 4: Payment Terms

In [8]:
query = "What are the payment terms?"
print(f"Query: {query}")
print("-" * 70)

try:
    results = agent.query_contracts(query, k=2)
    
    if results:
        for i, doc in enumerate(results, 1):
            print(f"\nResult {i}:")
            content = doc.page_content.strip()
            preview = content[:300] + "..." if len(content) > 300 else content
            print(preview)
    else:
        print("No results found")
        
except Exception as e:
    print(f"Query error: {str(e)}")

Query: What are the payment terms?
----------------------------------------------------------------------

Result 1:
nt new functionality or capability and does not constitute Value Add.
Term
The term of this TD will be Three (3) years from the Effective Date (“Initial Term”).
Invoicing and Ordering
Initial Invoice – only applies to Subscriptions
BP is placing the following order for the Cloud Services subscriptio...

Result 2:
nt new functionality or capability and does not constitute Value Add.
Term
The term of this TD will be Two (2) years from the Effective Date (“Initial Term”).
Invoicing and Ordering
Initial Invoice – only applies to Subscriptions
BP is placing the following order for the Cloud Services subscription(...


## 9. Semantic Search - Query 5: Termination Clauses

In [9]:
query = "Are there any termination clauses?"
print(f"Query: {query}")
print("-" * 70)

try:
    results = agent.query_contracts(query, k=2)
    
    if results:
        for i, doc in enumerate(results, 1):
            print(f"\nResult {i}:")
            content = doc.page_content.strip()
            preview = content[:300] + "..." if len(content) > 300 else content
            print(preview)
    else:
        print("No results found")
        
except Exception as e:
    print(f"Query error: {str(e)}")

Query: Are there any termination clauses?
----------------------------------------------------------------------

Result 1:
ncel it prior to expiration of the then current term, or the account will revert to pay-as-you-go. If an account reverts to pay-as-you-go, BP agrees to pay the applicable charges associated with such account as specified in the IBM invoice. The following link includes information on closing an accou...

Result 2:
ncel it prior to expiration of the then current term, or the account will revert to pay-as-you-go. If an account reverts to pay-as-you-go, BP agrees to pay the applicable charges associated with such account as specified in the IBM invoice. The following link includes information on closing an accou...


## 10. Process Additional Contracts (Optional)

Process remaining contracts to build a comprehensive contract database.

In [10]:
if len(contract_files) > 1:
    print(f"Processing {len(contract_files) - 1} additional contract(s)...\n")
    
    for contract_file in contract_files[1:]:
        print(f"Processing: {contract_file.name}")
        try:
            result = agent.run(str(contract_file.absolute()))
            print(f"   Successfully ingested")
        except Exception as e:
            print(f"   Error: {str(e)}")
    
    print(f"\nTotal contracts in vector database: {len(contract_files)}")
    print("   All contracts are now searchable via semantic queries")
else:
    print("Only one contract available. No additional contracts to process.")

Processing 4 additional contract(s)...

Processing: Confluent_IBM-1.30.2024.docx
DEBUG: Reading document from: /Users/arpit/Desktop/build engineering/sales-demo/agents/docs/Confluent_IBM-1.30.2024.docx
DEBUG: Extracted 11653 characters
DEBUG: Returning raw_text with 11653 characters
   Successfully ingested
Processing: Confluent_IBM-3.29.2024.docx
DEBUG: Reading document from: /Users/arpit/Desktop/build engineering/sales-demo/agents/docs/Confluent_IBM-3.29.2024.docx
DEBUG: Extracted 11674 characters
DEBUG: Returning raw_text with 11674 characters
   Successfully ingested
Processing: Confluent_IBM-5.30.2023.docx
DEBUG: Reading document from: /Users/arpit/Desktop/build engineering/sales-demo/agents/docs/Confluent_IBM-5.30.2023.docx
DEBUG: Extracted 11554 characters
DEBUG: Returning raw_text with 11554 characters
   Successfully ingested
Processing: Confluent Sales Cloud Infor.xlsx
DEBUG: Reading document from: /Users/arpit/Desktop/build engineering/sales-demo/agents/docs/Confluent Sales 

## 11. Custom Query

Try your own semantic search query on the ingested contracts.

In [11]:
# Enter your custom query here
custom_query = "What are the renewal terms?"  # Modify this

print(f"Custom Query: {custom_query}")
print("-" * 70)

try:
    results = agent.query_contracts(custom_query, k=3)
    
    if results:
        for i, doc in enumerate(results, 1):
            print(f"\nResult {i}:")
            content = doc.page_content.strip()
            preview = content[:300] + "..." if len(content) > 300 else content
            print(preview)
            print(f"\nSource: {doc.metadata.get('file_path', 'N/A')}")
    else:
        print("No results found")
        
except Exception as e:
    print(f"Query error: {str(e)}")

Custom Query: What are the renewal terms?
----------------------------------------------------------------------

Result 1:
h team on size needed to renew
	Automation Optimization	Kylie Brittz	Design	200000	2026-05-31 00:00:00	Apptio, Cloudability	Meeting with CTO and CPO is discuss cloud optimization they can do with Apptio and Cloudability
	watsonx ESA	Kylie Brittz	Engage	500000	2026-07-31 00:00:00	watsonx Orchestrate	...

Source: /Users/arpit/Desktop/build engineering/sales-demo/agents/docs/Confluent Sales Cloud Infor.xlsx

Result 2:
new functionality or capability and does not constitute Value Add.
Term
The term of this TD will be one (1) year from the Effective Date (“Initial Term”).
Invoicing and Ordering
Initial Invoice – only applies to Subscriptions
BP is placing the following order for the Cloud Services subscription(s) b...

Source: /Users/arpit/Desktop/build engineering/sales-demo/agents/docs/Confluent_IBM-1.30.2025.docx

Result 3:
new functionality or capability and does n

## Summary

### Demonstrated Capabilities
- Document reading (read-only access)
- OCR and text extraction from multiple formats
- Metadata extraction using IBM Granite LLM
- Contract data normalization
- Vector database ingestion with IBM embeddings
- Semantic search across contracts

### LangGraph Workflow
```
Read Document -> Extract Metadata -> Normalize -> Ingest Vector DB -> Generate Summary
```

### Vector Store
All contracts are stored in: `./contract_vector_store`

### Use Cases
- ESA/Contract ingestion and analysis
- Compliance checking
- Contract comparison
- Risk assessment
- Automated contract review

---